# Customer segmentation and retention analysis

### About Dataset

This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.

In [ ]:
!pip install xgboost

In [1]:
# import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
data = pd.read_csv(r"C:\Datasets\Online Retail.csv")
data.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


In [3]:
data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Drop Missing customer ID's

In [4]:
data = data.dropna(subset=["CustomerID"])

In [5]:
data["CustomerID"].isna().sum()

0

In [8]:
data.shape

(406829, 8)

# Data Cleaning

## Drop Canceled Transactions: All rows where InvoiceNo starts with "C" or where Quantity is negative

In [6]:
data = data[~data["InvoiceNo"].str.startswith("C")]
data = data[data["Quantity"] > 0]

In [7]:
data.shape

(397924, 8)

## Drop rows where UnitPrice is zero or negative

In [8]:
data = data[data["UnitPrice"] > 0]

In [9]:
data.shape

(397884, 8)

## Fill 'Description' null values with the corresponding StockCode.

In [10]:
data["Description"] = data["Description"].fillna(data["StockCode"])

In [11]:
data.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [12]:
# Data Distribution 
data.describe()

,Quantity,UnitPrice,CustomerID
count,397884.000000,397884.000000,397884.000000
mean,12.988238,3.116488,15294.423453
std,179.331775,22.097877,1713.141560
min,1.000000,0.001000,12346.000000
25%,2.000000,1.250000,13969.000000
50%,6.000000,1.950000,15159.000000
75%,12.000000,3.750000,16795.000000
max,80995.000000,8142.750000,18287.000000


In [13]:
data.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

### Validate Date data Type Column

In [14]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

# 1. Churn Definition 

In [15]:
# Calculate last purchase Date
customer_level_df = data.groupby("CustomerID")["InvoiceDate"].max().reset_index()
customer_level_df.columns = ["CustomerID","LastPurchaseDate"]

In [16]:
customer_level_df.head()

,CustomerID,LastPurchaseDate
0,12346.0,2011-01-18 10:01:00
1,12347.0,2011-12-07 15:52:00
2,12348.0,2011-09-25 13:13:00
3,12349.0,2011-11-21 09:51:00
4,12350.0,2011-02-02 16:01:00


In [17]:
customer_level_df.shape

(4338, 2)

In [18]:
reference_date = data["InvoiceDate"].max()

In [19]:
reference_date

Timestamp('2011-12-09 12:50:00')

In [20]:
customer_level_df["recency"] = reference_date - customer_level_df["LastPurchaseDate"]

In [21]:
customer_level_df

,CustomerID,LastPurchaseDate,recency
0,12346.0,2011-01-18 10:01:00,325 days 02:49:00
1,12347.0,2011-12-07 15:52:00,1 days 20:58:00
2,12348.0,2011-09-25 13:13:00,74 days 23:37:00
3,12349.0,2011-11-21 09:51:00,18 days 02:59:00
4,12350.0,2011-02-02 16:01:00,309 days 20:49:00
...,...,...,...
4333,18280.0,2011-03-07 09:52:00,277 days 02:58:00
4334,18281.0,2011-06-12 10:53:00,180 days 01:57:00
4335,18282.0,2011-12-02 11:43:00,7 days 01:07:00
4336,18283.0,2011-12-06 12:02:00,3 days 00:48:00


In [22]:
customer_level_df["recency"] = customer_level_df["recency"].dt.days

In [23]:
customer_level_df.head()

,CustomerID,LastPurchaseDate,recency
0,12346.0,2011-01-18 10:01:00,325
1,12347.0,2011-12-07 15:52:00,1
2,12348.0,2011-09-25 13:13:00,74
3,12349.0,2011-11-21 09:51:00,18
4,12350.0,2011-02-02 16:01:00,309


## Convert CustomerID datatype to integer from float type

In [24]:
customer_level_df["CustomerID"]= customer_level_df["CustomerID"].astype('int64')

In [25]:
customer_level_df.dtypes

CustomerID                   int64
LastPurchaseDate    datetime64[ns]
recency                      int64
dtype: object

In [26]:
customer_level_df.head(3)

,CustomerID,LastPurchaseDate,recency
0,12346,2011-01-18 10:01:00,325
1,12347,2011-12-07 15:52:00,1
2,12348,2011-09-25 13:13:00,74


In [27]:
def churn_label(recency):
    if recency > 90:
        return 1
    else:
        return 0
    
customer_level_df["churn_status"] = customer_level_df["recency"].apply(churn_label)

In [28]:
customer_level_df.head(3)

,CustomerID,LastPurchaseDate,recency,churn_status
0,12346,2011-01-18 10:01:00,325,1
1,12347,2011-12-07 15:52:00,1,0
2,12348,2011-09-25 13:13:00,74,0


In [29]:
customer_level_df.dtypes

CustomerID                   int64
LastPurchaseDate    datetime64[ns]
recency                      int64
churn_status                 int64
dtype: object

In [30]:
customer_level_df["churn_status"].value_counts()

0    2893
1    1445
Name: churn_status, dtype: int64

## Calculate Frequency

In [31]:
frequency = data.groupby("CustomerID")["InvoiceNo"].nunique().reset_index(name="Frequency")

In [32]:
customer_level_df = pd.merge(customer_level_df,frequency, on="CustomerID")

In [33]:
customer_level_df

,CustomerID,LastPurchaseDate,recency,churn_status,Frequency
0,12346,2011-01-18 10:01:00,325,1,1
1,12347,2011-12-07 15:52:00,1,0,7
2,12348,2011-09-25 13:13:00,74,0,4
3,12349,2011-11-21 09:51:00,18,0,1
4,12350,2011-02-02 16:01:00,309,1,1
...,...,...,...,...,...
4333,18280,2011-03-07 09:52:00,277,1,1
4334,18281,2011-06-12 10:53:00,180,1,1
4335,18282,2011-12-02 11:43:00,7,0,2
4336,18283,2011-12-06 12:02:00,3,0,16


## Calculate Monetory value per customer

In [34]:
revenue_per_transaction = data["Quantity"] * data["UnitPrice"]

In [35]:
data["Revenue"] = revenue_per_transaction

In [36]:
customer_value = data.groupby("CustomerID")["Revenue"].sum().reset_index(name="Customer_value")

In [37]:
customer_level_df = pd.merge(customer_level_df,customer_value, on="CustomerID")

In [38]:
customer_level_df

,CustomerID,LastPurchaseDate,recency,churn_status,Frequency,Customer_value
0,12346,2011-01-18 10:01:00,325,1,1,77183.60
1,12347,2011-12-07 15:52:00,1,0,7,4310.00
2,12348,2011-09-25 13:13:00,74,0,4,1797.24
3,12349,2011-11-21 09:51:00,18,0,1,1757.55
4,12350,2011-02-02 16:01:00,309,1,1,334.40
...,...,...,...,...,...,...
4333,18280,2011-03-07 09:52:00,277,1,1,180.60
4334,18281,2011-06-12 10:53:00,180,1,1,80.82
4335,18282,2011-12-02 11:43:00,7,0,2,178.05
4336,18283,2011-12-06 12:02:00,3,0,16,2094.88


## 2. Calculate 'Customer Lifetime Value'
### CLV = Average Order Value × Purchase Frequency × Customer Lifespan

#### - Purchase Frequency  = Frequency 
#### - Average Order Value = Customer_value ÷ Frequency
#### - Customer Lifespan = Last Purchase Date — First Purchase Date

In [39]:
# Calculate Customer Lifespan.
firstPurchaseDate = data.groupby("CustomerID")["InvoiceDate"].min().reset_index(name="FirstPurchaseDate")

In [40]:
customer_level_df = pd.merge(customer_level_df,firstPurchaseDate,on="CustomerID")

In [41]:
customer_level_df["CustomerLifespan"] = (customer_level_df["LastPurchaseDate"] - customer_level_df["FirstPurchaseDate"]).dt.days

In [42]:
customer_level_df = customer_level_df.drop(columns=["LastPurchaseDate","FirstPurchaseDate"])

In [43]:
customer_level_df.head(3)

,CustomerID,recency,churn_status,Frequency,Customer_value,CustomerLifespan
0,12346,325,1,1,77183.60,0
1,12347,1,0,7,4310.00,365
2,12348,74,0,4,1797.24,282


#### CLV = (Customer_value ÷ Frequency) × Frequency × CustomerLifespan
#### CLV = Customer_value X CustomerLifespan

In [44]:
customer_level_df["customer_lifetime_value"] = customer_level_df["Customer_value"] * customer_level_df["CustomerLifespan"]

In [45]:
customer_level_df.head(3)

,CustomerID,recency,churn_status,Frequency,Customer_value,CustomerLifespan,customer_lifetime_value
0,12346,325,1,1,77183.60,0,0.00
1,12347,1,0,7,4310.00,365,1573150.00
2,12348,74,0,4,1797.24,282,506821.68


### Interesting Stories in This Data
Customer 12346 — CLV = 0
Despite spending 77,183.60 their CLV is zero because their lifespan is 0 days — bought once and never returned. Confirms our bulk buyer theory. High spender but zero long term value.<br>
Customer 12347 — Highest CLV<br>
Active for a full 365 days, buying 7 times — this is exactly the kind of loyal customer a business wants to retain at all costs.<br>
Customer 12348 — Solid CLV<br>
Good frequency, decent lifespan — a reliable mid-tier customer.

## 3. Customer RFM Score

In [46]:
# Customer RFM
customer_level_df['R_score'] = pd.qcut(customer_level_df['recency'],q=4, labels=[4,3,2,1])
customer_level_df['R_score'] = customer_level_df['R_score'].astype('int64')

In [47]:
customer_level_df['F_score'] = pd.cut(customer_level_df['Frequency'],bins=[0, 1, 3, 6, float('inf')],
    labels=[1, 2, 3, 4])

In [48]:
customer_level_df['F_score'] = customer_level_df['F_score'].astype('int64')

In [49]:
customer_level_df['M_score'] = pd.qcut(customer_level_df['Customer_value'],q=4,labels=[1,2,3,4])
customer_level_df['M_score'] = customer_level_df['M_score'].astype('int64')

#### RFM_Score = R_score + F_score + M_score

In [50]:
customer_level_df["RFM_score"] = customer_level_df['R_score'] + customer_level_df['F_score'] + customer_level_df['M_score']

In [51]:
customer_level_df.head(3)

,CustomerID,recency,churn_status,Frequency,Customer_value,CustomerLifespan,customer_lifetime_value,R_score,F_score,M_score,RFM_score
0,12346,325,1,1,77183.60,0,0.00,1,1,4,6
1,12347,1,0,7,4310.00,365,1573150.00,4,4,4,12
2,12348,74,0,4,1797.24,282,506821.68,2,3,4,9


### The Stories Are Fascinating
Customer 12347 — Perfect Score 12 🏆

Bought recently, buys frequently, spends well
This is your Champion customer — highest priority to retain

Customer 12348 — Score 9 ⭐

Good recency, decent frequency and spend
Loyal customer — worth nurturing

Customer 12346 — Score 6 ⚠️

High monetary but churned, never returned
One time bulk buyer — high spend but no loyalty

## 4. Churn Prediction Model

In [52]:
features = [
    'Frequency', 
    'Customer_value',
    'CustomerLifespan',
    'customer_lifetime_value'
]

target = 'churn_status'

In [53]:
X = customer_level_df[features]
y = customer_level_df[target]

In [59]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [62]:
!pip install xgboost

In [64]:
#model = RandomForestClassifier(class_weight='balanced', n_estimators=500,random_state=42)
model = XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
    random_state=42
)

In [65]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [66]:
y_pred = model.predict(X_test)

In [68]:
accuracy_score(y_test, y_pred)

0.6970046082949308

In [69]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.67      0.74       561
           1       0.55      0.75      0.64       307

    accuracy                           0.70       868
   macro avg       0.69      0.71      0.69       868
weighted avg       0.73      0.70      0.70       868



In [70]:
print(confusion_matrix(y_test, y_pred))

[[376 185]
 [ 78 229]]


In [73]:
print(y.value_counts())

0    2893
1    1445
Name: churn_status, dtype: int64


## Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)
print(importance)